# Add Runoff

River runoff adds fresh-water discharge at the ocean surface.
CrocoDash supports two routes:

- **[Section A](#section-a-glofas-or-jra-runoff-via-compset)** — use GLOFAS or JRA runoff that ships with CESM (recommended starting point).
- **[Section B](#section-b-add-your-own-runoff-product-advanced)** — plug in a custom runoff dataset (advanced).

## Section A: GLOFAS or JRA Runoff via Compset

Both GLOFAS and JRA runoff products ship with CESM. To activate either:

1. Choose a compset alias that includes runoff (e.g. `CR_JRA_GLOFAS`).
2. Pass the ESMF mesh file for your chosen dataset to `configure_forcings` so
   CrocoDash can compute the remapping weights.

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="NCGD0011",
    override=True,
    machine="derecho",
    compset="CR_JRA_GLOFAS",  # swap _GLOFAS for _JRA if preferred
)

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    runoff_esmf_mesh_filepath="<ESMF_MESH_PATH>",
)

```{note}
On **Derecho/Casper** the mesh files are already available:

```{code-block} python
# GLOFAS
runoff_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/cseg/inputdata/ocn/mom/croc/rof/glofas/dis24/GLOFAS_esmf_mesh_v4.nc"
# JRA
runoff_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/inputdata/lnd/dlnd7/JRA55/JRA.v1.4.runoff.1958_ESMFmesh_cdf5_20201020.nc"
```
```

## Section B: Add Your Own Runoff Product (Advanced)

To use a custom runoff dataset you need three things: an ESMF mesh file, a
stream definition file, and the grid dimensions of your raw data.

### Step 1: Create an ESMF Mesh File

Generate a mesh file from a `Grid` that matches your raw data's grid:

In [ ]:
from CrocoDash.grid import Grid
from CrocoDash.topo import Topo

grid = Grid(
    lenx=360,
    leny=150,
    cyclic_x=True,
    ystart=-60,
    resolution=0.10,
    name="GLOFAS",
)
topo = Topo(grid, min_depth=0)
topo.set_flat(10)
topo.write_esmf_mesh("<path>")

### Step 2: Stream Definition File

In your case directory, create `drof.streams.xml` with the content below,
replacing all `<PLACEHOLDER>` values with your actual paths and variable names:

```xml
<?xml version="1.0"?>
<file id="stream" version="2.0">
  <stream_info name="rof.<PRODUCT_NAME>">
    <taxmode>cycle</taxmode>
    <tintalgo>upper</tintalgo>
    <readmode>single</readmode>
    <mapalgo>bilinear</mapalgo>
    <dtlimit>3.0</dtlimit>
    <year_first>START_YEAR</year_first>
    <year_last>END_YEAR</year_last>
    <year_align>START_YEAR</year_align>
    <vectors>null</vectors>
    <meshfile>PATH_TO_ESMF_MESH_FILE</meshfile>
    <lev_dimname>null</lev_dimname>
    <datafiles>
      <file>PATH_TO_RAW_DATA_NETCDF3_64BIT_OFFSET</file>
    </datafiles>
    <datavars>
      <var>NETCDF_VARIABLE_NAME Forr_rofl</var>
    </datavars>
    <offset>0</offset>
  </stream_info>
</file>
```

Also register your product in `components/cdeps/drof/namelist_definition_drof.xml`
and `config_component` — see the CESM docs for details.

### Step 3: Set Grid Dimensions

Run these `xmlchange` commands in your case directory:

```bash
./xmlchange ROF_NY=1500
./xmlchange ROF_NX=3600
./xmlchange ROF_DOMAIN_MESH=<MESH_PATH>
```

Then pass your mesh file to `configure_forcings` as shown in Section A so
CrocoDash generates the ocean remapping weights.